# Xenium MM samples — import and analysis

Downloads the 6 Xenium outputs that share block IDs with the Visium samples in
`ALL_MM_01July2026_fixed.ipynb`, then runs the full joint-analysis pipeline:
concatenation, normalization, PCA, Harmony batch correction, UMAP, Leiden clustering,
cell-type annotation, and spatial visualization.

**Data workflow:**
1. Download each Xenium output ZIP from HISE using `hp.cache_files()`.
2. Extract the ZIP into `xenium/data/outs/<name>/`.
3. Read with `spatialdata_io.xenium()` — the standard reader for 10x Xenium output
   directories. It returns a `SpatialData` object containing the cell expression table,
   morphology images, and cell/nucleus boundaries, all registered in a shared coordinate
   system.

In [ ]:
import os, shutil, zipfile, tempfile, warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq
import anndata as ad
import harmonypy as hm
import hisepy as hp
from spatialdata_io import xenium

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

In [ ]:
OUTS_DIR = '/home/workspace/PCD_data/xenium/data/outs'
os.makedirs(OUTS_DIR, exist_ok=True)

FIGURES_DIR = '../figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
sc.settings.figdir = FIGURES_DIR
plt.rcParams['savefig.dpi'] = 300

## HISE UUID map

UUIDs come from the `hise uuid` column of the `main` tab in
`xenium/data/xenium data tracking.xlsx`, matched by block ID.

TIS05397 has two Xenium sections from the same block. Section 008 (2024-01-30, Normal BM)
is the one carried forward as sk6.

In [ ]:
XENIUM_SAMPLES = {
    # name (→ OUTS_DIR/<name>/)   HISE UUID                               section   disease
    'TIS05393':     '70f7c0d6-8d7e-4b86-acf3-ca80137fd849',  # 009     Multiple Myeloma
    'TIS04788':     '38fe47a0-c09f-4373-9a8a-8a957c40aa9f',  # 010     Multiple Myeloma
    'TIS05392':     '0c879d26-970f-47fc-a705-88b103b4de80',  # 007     Multiple Myeloma
    'TIS05394':     'd7b4b14e-9856-4169-a363-213b4e29a99e',  # 008     Multiple Myeloma
    'TIS04789':     '7c3113c3-23bf-4840-8f45-61505048322c',  # 017     Multiple Myeloma
    'TIS05397_008': '16752455-bd23-4f88-8d05-2108e7778496',  # 008     Normal BM (2024)
}

## Download and extract

`hp.cache_files()` downloads the Xenium output ZIP from HISE into the read-only
`~/input/` cache and returns the local path to the ZIP file.

Inside each ZIP is one folder (e.g.
`output-XETG00123__0010984__TIS05392-001-007__20240130__235849`). That folder is moved
to `OUTS_DIR/<name>/` so `spatialdata_io.xenium()` can find the standard Xenium output
files at a predictable path.

If `OUTS_DIR/<name>/` already exists the download is skipped, so re-running is safe.

In [ ]:
def download_xenium(name, uuid, outs_dir):
    dest = os.path.join(outs_dir, name)
    if os.path.isdir(dest):
        print(f'{name}: already extracted — skipping')
        return

    print(f'{name}: downloading from HISE (UUID {uuid})…')
    (zip_path,) = hp.cache_files(file_ids=[uuid])

    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)

        # The ZIP unpacks to one top-level output folder; move it to dest.
        inner = next(
            os.path.join(tmp, d) for d in os.listdir(tmp)
            if os.path.isdir(os.path.join(tmp, d))
        )
        shutil.move(inner, dest)

    print(f'{name}: extracted → {dest}')


for name, uuid in XENIUM_SAMPLES.items():
    download_xenium(name, uuid, OUTS_DIR)

## Read Xenium data

`spatialdata_io.xenium(path)` reads a Xenium output directory and returns a `SpatialData`
object — a container that holds all layers of the experiment in a single coordinate system:

- **`sdata.tables['table']`** — AnnData with the gene x cell count matrix. `obs_names`
  are cell IDs; `obs` carries per-cell metadata (area, transcript counts, etc.).
- **`sdata.images`** — morphology images (DAPI focus or H&E), registered to the same
  coordinate space as the cells.
- **`sdata.shapes`** — cell and nucleus boundary polygons.

`obsm['spatial']` is populated directly by `spatialdata_io` — it holds each cell's x/y
centroid in microns. This is the coordinate array that `sc.pl.embedding(basis='spatial')`
reads for spatial plots.

In [ ]:
def read_xenium(name, outs_dir):
    path = os.path.join(outs_dir, name)
    sdata = xenium(path)
    adata = sdata.tables['table']
    # spatialdata_io populates obsm['spatial'] directly; no centroid columns in obs.
    return adata


sk1 = read_xenium('TIS05393', OUTS_DIR)
sk2 = read_xenium('TIS04788', OUTS_DIR)
sk3 = read_xenium('TIS05392', OUTS_DIR)
sk4 = read_xenium('TIS05394', OUTS_DIR)
sk5 = read_xenium('TIS04789', OUTS_DIR)
sk6 = read_xenium('TIS05397_008', OUTS_DIR)

for name, adata in [
    ('TIS05393', sk1), ('TIS04788', sk2), ('TIS05392', sk3),
    ('TIS05394', sk4), ('TIS04789', sk5), ('TIS05397_008', sk6),
]:
    print(f'{name}: {adata.n_obs:,} cells, {adata.n_vars} genes')

## Sample verification

Before combining, inspect each sample with two plots adapted from the Visium QC notebook.

**What `n_genes_by_counts` means.** The number of distinct genes detected at least once
in that cell. A cell with 40 counts spread across 38 genes is different from one with 40
counts all from 2 genes, and this column separates them.

### Per-sample distributions

Three views of each sample, one row per sample.

**Left and middle — histograms on a log axis.** The x-axis is `log10(value + 1)` rather
than the raw value: most cells have small counts and a few have very large ones, so on a
plain axis the shape is invisible. The `+ 1` exists because `log10(0)` is undefined and
some cells genuinely have zero.

**What to look for: one hump or two.** One hump means cells form a single population.
Two humps separated by a dip means two populations — cells that captured RNA and cells
that essentially did not. The dip is a natural place to apply a quality cutoff.

**Right — the knee curve.** Sort every cell from highest count to lowest and plot count
against rank, both axes logarithmic. A flat stretch on the left (well-captured cells),
then a bend, then a steep drop through near-empty cells. The bend is the "knee" and marks
where good captures end.

### Spatial QC maps

The histograms show *how many* cells have a given value. They cannot show *where* those
cells sit on the tissue. The spatial maps do.

Cells are plotted at their x/y centroids in microns, coloured by each QC metric. The
colour scale is clipped at the 99th percentile (`vmax='p99'`) so a handful of
high-value cells do not compress the rest of the range into one shade.

**What to look for.** A smooth gradient across the section suggests a technical artifact.
A thin rim of low values at the tissue edge is normal — edge cells are partly outside the
capture area. A solid low-count patch in the middle of the tissue is the thing to
investigate before filtering.

In [ ]:
samples_dict = {
    'TIS05393': sk1, 'TIS04788': sk2, 'TIS05392': sk3,
    'TIS05394': sk4, 'TIS04789': sk5, 'TIS05397_008': sk6,
}

# n_genes_by_counts: genes with at least one count per cell
for adata in samples_dict.values():
    adata.obs['n_genes_by_counts'] = np.asarray((adata.X > 0).sum(axis=1)).flatten()

# ── Per-sample distributions ──────────────────────────────────────────────────
fig, axes = plt.subplots(len(samples_dict), 3, figsize=(15, 3.2 * len(samples_dict)), squeeze=False)

for row_axes, (name, adata) in zip(axes, samples_dict.items()):
    counts = adata.obs['total_counts'].to_numpy()
    genes  = adata.obs['n_genes_by_counts'].to_numpy()

    row_axes[0].hist(np.log10(counts + 1), bins=60, color='steelblue')
    row_axes[0].set_xlabel('log10(counts + 1)')
    row_axes[0].set_title(f'{name} — counts  (median {np.median(counts):.0f})')

    row_axes[1].hist(np.log10(genes + 1), bins=60, color='steelblue')
    row_axes[1].set_xlabel('log10(genes + 1)')
    row_axes[1].set_title(f'{name} — genes  (median {np.median(genes):.0f})')

    row_axes[2].plot(np.arange(1, len(counts) + 1), np.sort(counts)[::-1], lw=1.5)
    row_axes[2].set_xscale('log')
    row_axes[2].set_yscale('log')
    row_axes[2].set_xlabel('cell rank')
    row_axes[2].set_ylabel('counts')
    row_axes[2].set_title(f'{name} — knee')

fig.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'qc_distributions.{ext}'), dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    name: {
        'n_cells':       adata.n_obs,
        'median_counts': int(np.median(adata.obs['total_counts'])),
        'median_genes':  int(np.median(adata.obs['n_genes_by_counts'])),
    }
    for name, adata in samples_dict.items()
}).T
print(summary.to_string())

In [ ]:
QC_COLUMNS = ['total_counts', 'n_genes_by_counts']

fig, axes = plt.subplots(len(samples_dict), len(QC_COLUMNS),
                         figsize=(5.5 * len(QC_COLUMNS), 5 * len(samples_dict)), squeeze=False)

for i, (name, adata) in enumerate(samples_dict.items()):
    for j, column in enumerate(QC_COLUMNS):
        sc.pl.embedding(
            adata, basis='spatial', color=column, vmax='p99',
            ax=axes[i, j], show=False, 
            title=f'{name} — {column}  (median {adata.obs[column].median():.0f})',
        )

fig.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'spatial_qc_maps.{ext}'), dpi=150, bbox_inches='tight')
plt.show()

## Combine samples

Each sample is an independent `AnnData` — one row per cell. Before any joint analysis
we concatenate them into a single object and tag each cell with its sample of origin.

`obs_names_make_unique()` appends a suffix to each cell ID so there are no duplicate row
labels after concatenation.

`join='inner'` keeps only genes present in all six samples. All share the same 377-gene
Xenium panel, so nothing is dropped.

In [ ]:
sk1.obs_names_make_unique()
sk2.obs_names_make_unique()
sk3.obs_names_make_unique()
sk4.obs_names_make_unique()
sk5.obs_names_make_unique()
sk6.obs_names_make_unique()

combined = sc.concat(
    {'TIS05393': sk1, 'TIS04788': sk2, 'TIS05392': sk3,
     'TIS05394': sk4, 'TIS04789': sk5, 'TIS05397_008': sk6},
    label='sample',
    join='inner',
)
combined.obs_names_make_unique()

print(f'{combined.n_obs:,} cells x {combined.n_vars} genes')
print(combined.obs['sample'].value_counts())

## Preprocessing

Split into two steps so the filter can be tuned before the expensive computation runs.

**Step 1 — QC metrics and cell filtering.**
Raw Xenium output includes many near-empty cell detections from the segmentation
algorithm. The 6 samples total 399K cells; the previous h5ad analysis of the same tissue
had 267K, meaning roughly 130K of these are low-quality. Removing them before PCA and
UMAP is what prevents memory exhaustion.

`MIN_COUNTS` and `MIN_GENES` are conservative starting points. Adjust them based on the
histograms in the verification section: look for the dip between the two humps, or the
knee of the rank curve.

**Step 2 — normalization and dimensionality reduction.**
Runs on the filtered cells only.
1. **Save raw counts** in `layers["counts"]` before any transformation.
2. **Normalize** each cell to 10,000 total counts.
3. **PCA** — 30 components across all 377 genes.
4. **k-nearest neighbors** graph (k = 15) in PCA space.
5. **UMAP** — 2-D embedding.
6. **Leiden clustering** — community detection.

Note: log-normalization (`sc.pp.log1p`) is intentionally skipped, matching the original
analysis.

In [ ]:
# percent_top default is (50, 100, 200, 500), but 500 > 377 genes in the Xenium panel.
sc.pp.calculate_qc_metrics(combined, percent_top=None, inplace=True)

# ── Cell filter ───────────────────────────────────────────────────────────────
# Adjust these based on the histograms above: look for the dip between the two
# humps (if present) or the knee of the rank curve.
MIN_COUNTS = 10
MIN_GENES  = 5

n_before = combined.n_obs
sc.pp.filter_cells(combined, min_counts=MIN_COUNTS)
sc.pp.filter_cells(combined, min_genes=MIN_GENES)
n_after = combined.n_obs

print(f'Filtered {n_before - n_after:,} cells  '
      f'({100 * (n_before - n_after) / n_before:.1f}% removed)')
print(f'Remaining: {n_after:,} cells')
print(combined.obs['sample'].value_counts())

In [ ]:
combined.layers['counts'] = combined.X.copy()
sc.pp.normalize_total(combined, inplace=True)

if not sp.issparse(combined.X):
    combined.X = sp.csr_matrix(combined.X)

sc.pp.pca(combined, n_comps=30, mask_var=None)
sc.pp.neighbors(combined, n_neighbors=15, n_pcs=30)
sc.tl.umap(combined)
sc.tl.leiden(combined, flavor='igraph', n_iterations=2, directed=False, resolution=0.8)

In [ ]:
sc.pl.umap(
    combined,
    color=['total_counts', 'n_genes_by_counts', 'leiden', 'sample'],
    wspace=0.4,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'umap_pre_harmony_UMAP.{ext}'), bbox_inches='tight')
plt.show()

In [ ]:
ho = hm.run_harmony(
    combined.obsm['X_pca'].astype('float64'),
    combined.obs,
    'sample',
)
z = ho.Z_corr
if z.shape[0] != combined.n_obs:
    z = z.T
combined.obsm['X_pca_harmony'] = z
combined.obsm['X_pca'] = combined.obsm['X_pca_harmony']

sc.pp.neighbors(combined, use_rep='X_pca')
sc.tl.umap(combined)
sc.tl.rank_genes_groups(combined, 'leiden', method='wilcoxon')

In [ ]:
sc.pl.umap(
    combined,
    color=['total_counts', 'n_genes_by_counts', 'leiden', 'sample'],
    wspace=0.4,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'umap_harmony_UMAP_MM.{ext}'), bbox_inches='tight')
plt.show()

sc.pl.rank_genes_groups(combined, n_genes=10, sharey=False, fontsize=20, show=False)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'rank_genes_groups.{ext}'), bbox_inches='tight')
plt.show()

## Cell type annotation

Cluster labels below come from a previous run on pre-processed h5ad files for the same
6 samples. The data loaded here comes directly from the raw Xenium output — more cells,
no pre-filtering — so Leiden will produce different cluster numbering.

**This mapping is a starting point, not a final annotation.** Compare the marker genes
from `rank_genes_groups` above to known biology and update the dictionary before
treating these labels as ground truth.

In [ ]:
cluster_to_cell_type = {
    '0': 'Erythroblast_1',
    '1': 'Macrophage',
    '2': 'Monocyte',
    '3': 'Endothelial_1',
    '4': 'Monocyte/Macrophage',
    '5': 'Stromal/Fibroblast',
    '6': 'Tumor-Associated Macrophage',
    '7': 'Neoplasm_1',
    '8': 'Erythroblast_2',
    '9': 'Neutrophil',
    '10': 'Neoplasm_2',
    '11': 'Neoplasm_3',
    '12': 'Cytotoxic T Cell',
    '13': 'Neoplasm_4',
    '14': 'Myeloid',
    '15': 'Neoplasm_5',
    '16': 'Endothelial_2',
}
combined.obs['cell_type'] = combined.obs['leiden'].map(cluster_to_cell_type).astype('category')
print(combined.obs['cell_type'].value_counts())

In [ ]:
sc.pl.umap(
    combined,
    color='cell_type',
    legend_loc='right margin',
    legend_fontsize=8,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'umap_UMAP_celltypes.{ext}'), bbox_inches='tight')
plt.show()

## Spatial expression plots

Each cell is plotted at its x/y centroid in microns (coordinates from `obsm['spatial']`,
set by `spatialdata_io`). The three marker genes shown are:

- **MZB1** — marks plasma cells (antibody-secreting B cells)
- **PECAM1** — marks endothelial cells (blood vessel lining)
- **GNLY** — marks NK cells and cytotoxic T cells

These serve as landmarks to confirm the spatial structure is intact after loading.

In [ ]:
genes_to_plot = ['MZB1', 'PECAM1', 'GNLY']
samples = combined.obs['sample'].cat.categories

cmap = mpl.colors.ListedColormap(['cyan', 'magenta', 'orange', 'red']).with_extremes(under='0.8', over='red')
bounds = [3, 4, 8, 20]
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

fig, axes = plt.subplots(
    len(samples), len(genes_to_plot),
    figsize=(5 * len(genes_to_plot), 5 * len(samples)),
)
for i, sample in enumerate(samples):
    slide = combined[combined.obs['sample'] == sample]
    for j, gene in enumerate(genes_to_plot):
        ax = axes[i, j] if len(samples) > 1 else axes[j]
        sc.pl.spatial(slide, color=gene, ax=ax, spot_size=50, cmap=cmap, norm=norm, show=False)
        if i == 0:
            ax.set_title(gene)
        if j == 0:
            ax.set_ylabel(f'Slide {sample}', fontsize=10)

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'spatial_gene_expression.{ext}'), dpi=150, bbox_inches='tight')
plt.show()

## Spatial cluster maps by sample

One panel per sample, cells colored by Leiden cluster or cell type. This shows whether
the spatial organization of clusters is consistent across samples or varies with tissue
architecture.

In [ ]:
def plot_spatial_by_sample(adata, color, ncol=3, suptitle='', size=2, save=None, **kwargs):
    samples = adata.obs['sample'].cat.categories
    nrow = int(np.ceil(len(samples) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4 * nrow), dpi=150)
    axs = axs.flatten()
    for i, sample in enumerate(samples):
        sub = adata[adata.obs['sample'] == sample]
        # sc.pl.embedding(
        #     sub, basis='spatial', color=color,
        #     ax=axs[i], title=sample, show=False, size=size, **kwargs
        # )
        sq.pl.spatial_scatter(
            sub,
            library_id="spatial",
            shape=None,
            color=["leiden"],
            wspace=0.4,
            ax=axs[i],
            title=sample
        )
    for ax in axs[len(samples):]:
        ax.set_visible(False)
    if suptitle:
        plt.suptitle(suptitle, y=1.01)
    plt.tight_layout()
    if save:
        base = os.path.splitext(save)[0]
        for ext in ['png', 'pdf']:
            plt.savefig(os.path.join(sc.settings.figdir, f'{base}.{ext}'), dpi=150, bbox_inches='tight')
    plt.show()


plot_spatial_by_sample(combined, color='leiden', suptitle='Leiden clusters by sample',
                       save='spatial_leiden_by_sample.png')
# plot_spatial_by_sample(combined, color='cell_type', suptitle='Cell types by sample',
#                        save='spatial_celltypes_by_sample.png')